# 02 — Compression: Selective TOC and chunk selection

In notebook 01 you saw STM proxy a tiny response. Real upstream
MCP servers return much bigger payloads — full files, long API
responses, large directory listings — and those burn through
the agent's context window fast.

STM has **10 compression strategies**. This notebook focuses
on the most useful one for structured data: **selective
compression**, which returns a *table of contents* and lets
the agent request only the sections it needs via
`stm_proxy_select_chunks`.

**You will learn:**

- How STM turns a ~18 KB structured response into a ~1.5 KB TOC
- How to parse the TOC to discover section keys
- How to retrieve specific sections on demand
- How to measure the token savings

**Prereqs:** Notebook 01 completed (or re-run its setup).

## 1. Isolate state and register the `docfix` upstream

In [ ]:
# Add notebooks/ to sys.path so `_helpers` is importable regardless of
# where Jupyter was launched (from the repo root or from notebooks/).
import sys
from pathlib import Path

_cwd = Path.cwd()
if (_cwd / "_helpers.py").exists():
    sys.path.insert(0, str(_cwd))
elif (_cwd / "notebooks" / "_helpers.py").exists():
    sys.path.insert(0, str(_cwd / "notebooks"))
else:
    raise RuntimeError(
        "Cannot find notebooks/_helpers.py — run Jupyter from the repo "
        "root or from the notebooks/ directory."
    )

In [ ]:
import json
import subprocess

from _helpers import (
    isolate_stm_state,
    stm_session,
    extract_text,
    fixtures_dir,
    parse_toc_response,
    token_estimate,
)

config_path = isolate_stm_state(prefix="nb02_")
doc_script = fixtures_dir() / "doc_mcp.py"

# --compression selective forces STM to use the selective
# compressor for this upstream regardless of content shape.
# --max-chars sets a tight budget (1000 chars) so even a
# medium-sized response triggers TOC generation.
subprocess.run(
    [
        "uv", "run", "mms", "add", "docfix",
        "--config", str(config_path),
        "--command", "uv",
        "--args", f"run python {doc_script}",
        "--prefix", "docfix",
        "--compression", "selective",
        "--max-chars", "1000",
    ],
    check=True, capture_output=True,
)
print("Registered docfix (selective, max 1000 chars)")

## 2. Call the tool and capture the TOC

The `docfix__get_document` tool returns a JSON object with 8
labeled sections totaling ~18 KB. Without STM the agent would
have to eat all of that. With STM in the middle, the selective
compressor intercepts the response and returns a compact TOC.

In [ ]:
async with stm_session() as session:
    result = await session.call_tool("docfix__get_document", {})
    raw_text = extract_text(result)

print(f"Response length: {len(raw_text)} chars  (~{token_estimate(raw_text)} tokens)")
print()
print(raw_text[:500])
print("...")

In [ ]:
# Parse the TOC and show its structure
toc = parse_toc_response(raw_text)
if toc is None:
    raise RuntimeError("Expected a TOC response — check compression config")

print(f"Type:          {toc['type']}")
print(f"Selection key: {toc['selection_key']}")
print(f"Total chars:   {toc.get('total_chars', 'n/a')}  (before compression)")
print(f"Entries ({len(toc['entries'])}):")
for entry in toc["entries"]:
    size = entry.get("size", 0)
    key = entry.get("key", "?")
    print(f"  - {key:<15} ({size} chars)")

## 3. Retrieve selected sections

Now the agent has enough information to pick what it actually
needs. `stm_proxy_select_chunks` takes the `selection_key` from
the TOC and a list of section keys and returns just those
sections — zero information loss for the parts you pick,
zero tokens spent on the rest.

In [ ]:
picked_sections = ["Selective", "Progressive"]

async with stm_session() as session:
    select_result = await session.call_tool(
        "stm_proxy_select_chunks",
        {"key": toc["selection_key"], "sections": picked_sections},
    )
    selected_text = extract_text(select_result)

print(f"Selected {len(picked_sections)} sections "
      f"({len(selected_text)} chars, ~{token_estimate(selected_text)} tokens)")
print()
print(selected_text[:800])
print("..." if len(selected_text) > 800 else "")

## 4. Compare: original vs TOC vs selected

Here's what just happened in token terms:

In [ ]:
original_chars = toc.get("total_chars", 0)
toc_chars = len(raw_text)
selected_chars = len(selected_text)

print("| Stage                       |    Chars |   ~Tokens |   Ratio |")
print("|-----------------------------|---------:|----------:|--------:|")
print(f"| Original document           | {original_chars:>8} | {token_estimate(str(original_chars)*original_chars)[:1] if False else token_estimate('x' * original_chars):>9} | 100.00% |")
print(f"| Compressed (TOC)            | {toc_chars:>8} | {token_estimate('x' * toc_chars):>9} | {toc_chars/original_chars*100:>5.2f}% |")
print(f"| Selected 2/8 sections       | {selected_chars:>8} | {token_estimate('x' * selected_chars):>9} | {selected_chars/original_chars*100:>5.2f}% |")
print()
savings = 100 * (1 - toc_chars / original_chars)
print(f"First-contact savings:     {savings:>5.1f}% — what the agent sees first")
print(f"After targeted retrieval:  {100 * (1 - selected_chars/original_chars):>5.1f}% savings vs the full doc")
print(f"                           (but 100% of the info it actually asked for)")

## Where to next

- **Progressive delivery** (`stm_proxy_read_more`) — for
  unstructured large responses that don't have natural
  sections, STM can chunk them and let the agent walk
  through with a cursor. See
  [`docs/compression.md`](../docs/compression.md).
- **The other 9 strategies** — auto-selection, hybrid,
  truncate, extract_fields, schema_pruning, skeleton,
  llm_summary, none — all explained in the same doc.
- **Notebook 03** — proactive memory surfacing: STM calling
  a long-term memory server and injecting relevant chunks
  into responses.